# Collaborative Filtering with ALS
This notebook trains a **user-item collaborative filter** on historical pizza-chain cart transactions using PySpark's Alternating Least Squares (ALS) implementation.

It shares a **common held-out test set** (`test_dataset`) with the MBA notebook (`01_market_basket_analysis`) so that Hit@k scores are directly comparable across both approaches.

### Compute Requirements
| | |
|---|---|
| **Runtime** | DBR 17.3 ML |
| **Compute** | Classic cluster |
| **Packages** | `optuna`, `databricks-feature-engineering` (installed in cell 2) |

| Section | Description |
|---|---|
| **1 — Data Preparation** | Load the train / test tables (saved by `00_data_preparation`), build user/item integer mappings, and compute implicit ratings |
| **2 — Model Construction** | Train a baseline ALS model, evaluate on a **validation** split, hyperparameter-tune with Optuna + MLflow, then train the final model on all training data and report Hit@k on the **held-out test set** |
| **3 — Deployment** | Pre-compute top-20 recommendations, publish to a Lakebase online store, and log a Lakebase-backed PyFunc for Model Serving |

### Data Splits
```
00_data_preparation outputs (all persisted as Delta tables):
  ├── cleaned_mapped_dataset   (full dataset — not used directly by this notebook)
  ├── train_dataset            (80 %, seed=42)
  │     ├── train_hp  (80 %)  → HPO trial training
  │     └── val_hp    (20 %)  → HPO trial evaluation
  └── test_dataset             (20 %, seed=42 — shared with MBA notebook)
```

### Serving Architecture
```
Frontend  →  Model Serving endpoint (ALSRecommenderModel PyFunc)
                 │
                 └── load_context: reads pre-computed recs from
                     Lakebase online table (als_recommendations)
                 │
                 └── predict(mpid) → ranked product recommendations
```

In [0]:
%pip install optuna --quiet
%pip install --upgrade databricks-feature-engineering --quiet
dbutils.library.restartPython()

In [0]:
catalog = 'users'
schema = 'jon_cheung'

# --- Input tables (from 00_data_preparation) ---
train_table          = f'{catalog}.{schema}.train_dataset'
test_table           = f'{catalog}.{schema}.test_dataset'

# --- Model & experiment names ---
experiment_name      = '/Users/jon.cheung@databricks.com/ALS_recommender_model'
model_name           = f'{catalog}.{schema}.als_recommender_model'

# --- Recommendations table ---
recs_table           = f'{catalog}.{schema}.als_recommendations'

# --- Online store ---
online_store_name    = 'pizza-chain-online-store'
online_store_fallback = 'demo-online-store'
online_table_name    = f'{catalog}.{schema}.als_recommendations_online'

In [0]:
from pyspark.sql.functions import col

# Load train and test tables saved by 00_data_preparation
train = spark.read.table(train_table)
test  = spark.read.table(test_table)

# Further split training data: 80 % for HPO training, 20 % for HPO validation
train_hp, val_hp = train.randomSplit([0.8, 0.2], seed=42)

print(f'{train.count():,} training  |  {test.count():,} test (shared with MBA notebook)')
print(f'{train_hp.count():,} HPO train  |  {val_hp.count():,} HPO validation')

## 1 — Data Preparation
1. Load `train_dataset` and `test_dataset` from Unity Catalog (saved by `00_data_preparation`)
2. Split training data into `train_hp` (80 %) and `val_hp` (20 %) for HPO
3. Create integer mappings for users and products (ALS requires numeric IDs)
4. Generate implicit “ratings” — the proportion of a user’s orders that contain each item
5. Prepare the validation and test sets for Hit@k evaluation

In [0]:
from pyspark.sql.functions import explode, col, count, sum as spark_sum, row_number
from pyspark.sql.window import Window

def preprocess_pipeline(dataset):
    """
    Convert raw orders into the (user_id, item_id, proportion_of_orders) format
    that ALS expects.

    Steps:
      1. Cast mpid to double
      2. Explode order_product_list → one row per (user, item)
      3. Create integer ID mappings for users and items
      4. Compute an implicit rating: proportion of a user’s total orders that
         include each item

    Returns:
      (als_dataset, item_mapping, user_mapping)
    """
    # 1. Cast user ID
    dataset = dataset.withColumn('mpid', col('mpid').cast('double'))

    # 2. Explode products
    sdf_exploded = dataset.withColumn('product', explode('order_product_list'))
    als_input = sdf_exploded.select(col('mpid').alias('user'), col('product').alias('item'))

    # 3. Integer ID mappings
    item_mapping = (als_input.select('item').distinct()
                    .withColumn('item_id', row_number().over(Window.orderBy('item'))))
    user_mapping = (als_input.select('user').distinct()
                    .withColumn('user_id', row_number().over(Window.orderBy('user'))))

    als_input_mapped = (
        als_input
        .join(item_mapping, on='item')
        .join(user_mapping, on='user')
        .select('user_id', 'item_id', 'item')
    )

    # 4. Implicit rating = proportion of user’s orders containing this item
    als_grouped = als_input_mapped.groupBy('user_id', 'item_id').agg(
        count('*').alias('historical_ordered_amount')
    )
    user_totals = als_grouped.groupBy('user_id').agg(
        spark_sum('historical_ordered_amount').alias('total_orders')
    )
    als_dataset = (
        als_grouped.join(user_totals, on='user_id')
        .withColumn('proportion_of_orders',
                    col('historical_ordered_amount') / col('total_orders'))
    )

    return als_dataset, item_mapping, user_mapping

In [0]:
# Preprocess the HPO training split (not the full training set)
als_train_dataset, item_mapping, user_mapping = preprocess_pipeline(train_hp)
print(f'HPO training matrix: {als_train_dataset.count():,} user-item pairs')
print(f'  Users:  {user_mapping.count():,}')
print(f'  Items:  {item_mapping.count():,}')

In [0]:
from pyspark.sql.functions import expr, size, col

def prepare_eval_set(dataset, user_mapping, item_mapping, label=''):
    """Hold out the last item in each order and link to training ID mappings."""
    filtered = dataset.filter(size(col('order_product_list')) > 1)
    transformed = (
        filtered
        .withColumn('cart', expr('slice(order_product_list, 1, size(order_product_list) - 1)'))
        .withColumn('added', expr('order_product_list[size(order_product_list) - 1]'))
        .withColumn('mpid', col('mpid').cast('double'))
    )
    linked = (
        transformed
        .join(user_mapping, transformed.mpid == user_mapping.user, 'inner')
        .join(item_mapping, transformed.added == item_mapping.item, 'inner')
        .withColumnRenamed('item_id', 'added_item_id')
    )
    total = dataset.count()
    print(f'{label}: {linked.count():,} linked ({total - linked.count():,} dropped out of {total:,})')
    return linked

# Validation set — used during HPO
val_linked = prepare_eval_set(val_hp, user_mapping, item_mapping, 'Validation')

# Test set — held out until final evaluation (same split as MBA notebook)
test_linked = prepare_eval_set(test, user_mapping, item_mapping, 'Test')


## 2 — Model Construction
Training pipeline:
1. Build a small (low-rank) ALS model to verify the preprocessing pipeline
2. Evaluate the baseline on the **validation set** (`val_hp`) using Hit@k
3. Hyperparameter-tune `rank` and `maxIter` with Optuna — each trial trains on `train_hp` and evaluates on `val_hp`
4. Train the final model on **all training data** (`train = train_hp + val_hp`) using the best hyperparameters
5. Report the final Hit@k on the **held-out test set** (same as MBA) and log to MLflow

In [0]:
from pyspark.ml.recommendation import ALS

def train_als(train_dataset, rank, maxIter):
    als = ALS(
        rank=rank,
        maxIter=maxIter,
        userCol='user_id',
        itemCol='item_id',
        ratingCol='proportion_of_orders',
        implicitPrefs=True
    )
    return als.fit(train_dataset)

In [0]:
model = train_als(als_train_dataset, rank=10, maxIter=5)
print('Small ALS model trained successfully (rank=10, maxIter=5)')

### Evaluate with Hit@k
Hit@k measures what proportion of the time the last item added to a test cart appears in the model’s top-k recommendations for that user. For example, Hit@5 = 0.25 means the held-out item is in the top 5 recommendations 25 % of the time.

In [0]:
from pyspark.sql.functions import array_contains, avg

k = 5

# Evaluate baseline model on the VALIDATION set (not test)
val_recommendations = model.recommendForUserSubset(
    val_linked.select('user_id').distinct(), k
)

val_recommendations = (
    val_recommendations
    .join(val_linked, 'user_id', 'inner')
    .select('mpid', 'user_id', 'cart', 'added', 'added_item_id', 'recommendations')
    .withColumn('hit_at_k',
                array_contains(col('recommendations.item_id'),
                               col('added_item_id')).cast('int'))
)

avg_hit_k = val_recommendations.agg(avg('hit_at_k').alias('avg_hit_at_k'))
print(f'Baseline ALS (rank=10, maxIter=5) — Hit@{k} on validation set:')
display(avg_hit_k)

### Hyperparameter tuning with Optuna + MLflow
ALS has two primary knobs:
* **`rank`** — dimensionality of the latent user/item factor matrices
* **`maxIter`** — number of alternating least-squares iterations until convergence

We use Optuna to search the space and log every trial as a nested MLflow run.

In [0]:
import mlflow
import optuna
from functools import partial
from pyspark.sql.functions import array_contains, avg, col

def objective(trial, parent_run_id, k=5):
    with mlflow.start_run(parent_run_id=parent_run_id, nested=True):
        params = {
            'rank': trial.suggest_int('rank', 1, 100),
            'maxIter': trial.suggest_int('maxIter', 1, 10)
        }

        model = train_als(als_train_dataset,
                          rank=params['rank'],
                          maxIter=params['maxIter'])

        # Evaluate on VALIDATION set (not test)
        val_recs = model.recommendForUserSubset(
            val_linked.select('user_id').distinct(), k
        )
        val_recs = (
            val_recs.join(val_linked, 'user_id', 'inner')
            .select('user_id', 'added_item_id', 'recommendations')
            .withColumn('hit_at_k',
                        array_contains(col('recommendations.item_id'),
                                       col('added_item_id')).cast('int'))
        )
        avg_hit = val_recs.agg(avg('hit_at_k')).first()[0]

        mlflow.log_params(params)
        mlflow.log_metric('avg_hit_at_k', avg_hit)

    return avg_hit

In [0]:
import os

os.makedirs('artifacts', exist_ok=True)
mlflow.set_experiment(experiment_name)
n_trials = 20

with mlflow.start_run(run_name='ALS_hp_tuning') as parent_run:
    # --- Optuna HPO (trains on train_hp, evaluates on val_hp) ---
    _objective = partial(objective,
                         parent_run_id=parent_run.info.run_id,
                         k=5)

    study = optuna.create_study(direction='maximize')
    study.optimize(_objective, n_trials=n_trials)

    best = study.best_trial
    print(f'Best trial Hit@5 (validation): {best.value:.4f}')
    print(f'Best params: {best.params}')

    # --- Train final model on ALL training data (train_hp + val_hp) ---
    with mlflow.start_run(run_name='final_als_model',
                          nested=True,
                          parent_run_id=parent_run.info.run_id):

        als_full_dataset, item_mapping_full, user_mapping_full = \
            preprocess_pipeline(train)

        final_model = train_als(als_full_dataset,
                                rank=best.params['rank'],
                                maxIter=best.params['maxIter'])

        # --- Final evaluation on HELD-OUT TEST SET (same as MBA) ---
        test_linked_final = prepare_eval_set(
            test, user_mapping_full, item_mapping_full, 'Final test')

        test_recs = final_model.recommendForUserSubset(
            test_linked_final.select('user_id').distinct(), 5)
        test_recs = (
            test_recs.join(test_linked_final, 'user_id', 'inner')
            .select('user_id', 'added_item_id', 'recommendations')
            .withColumn('hit_at_k',
                        array_contains(col('recommendations.item_id'),
                                       col('added_item_id')).cast('int'))
        )
        test_hit = test_recs.agg(avg('hit_at_k')).first()[0]
        mlflow.log_metric('test_hit_at_5', test_hit)
        print(f'Final model Hit@5 on TEST set: {test_hit:.4f}')

        # Log mappings as artifacts
        user_mapping_full.toPandas().to_parquet('artifacts/user_mapping_ALS.parquet')
        item_mapping_full.toPandas().to_parquet('artifacts/item_mapping_ALS.parquet')
        mlflow.log_artifact('artifacts/user_mapping_ALS.parquet', 'user_mapping')
        mlflow.log_artifact('artifacts/item_mapping_ALS.parquet', 'item_mapping')

        # Log the Spark ALS model
        mlflow.spark.log_model(final_model, artifact_path='model')
        mlflow.log_params(best.params)

        print(f'Final model logged (rank={best.params["rank"]}, '
              f'maxIter={best.params["maxIter"]})')

## 3 — Deployment
Three steps to go from a trained ALS model to a production-ready serving endpoint:

| Step | Cell | Description |
|---|---|---|
| **3a — Pre-compute** | Cell 18 | Use the final ALS model to generate top-20 ranked recommendations per user, reverse-map integer IDs back to product slugs, and save as `als_recommendations` Delta table |
| **3b — Lakebase publish** | Cells 19–20 | Enable CDF, add a primary-key constraint on `mpid`, and publish to a Lakebase online store for point-lookup serving |
| **3c — PyFunc model serving** | Cells 21–24 | Define `ALSRecommenderModel` (mirrors MBA’s `MBARecommenderModel`), log to MLflow / Unity Catalog with a config artifact, and test inference |

### How the frontend queries recommendations
```
POST /serving-endpoints/als_recommender_model/invocations
{"dataframe_records": [{"mpid": 42}]}

→ {"mpid": 42,
   "recommendations": ["pepperoni-pizza", "crazy-bread", ...],
   "scores": [0.98, 0.91, ...]}
```

In [0]:
from pyspark.sql.functions import col, explode, collect_list, struct, row_number
from pyspark.sql.window import Window

# Generate top-20 recommendations for every user in the training set
raw_recs = final_model.recommendForAllUsers(20)

# Explode the nested recommendations array and reverse-map IDs → product slugs
recs_exploded = (
    raw_recs
    .select('user_id', explode('recommendations').alias('rec'))
    .select('user_id',
            col('rec.item_id').alias('item_id'),
            col('rec.rating').alias('score'))
)

recs_mapped = (
    recs_exploded
    .join(item_mapping_full, on='item_id')
    .join(user_mapping_full, on='user_id')
    .select(
        col('user').cast('long').alias('mpid'),
        'item', 'score'
    )
)

# Rank within each user and collect into ordered arrays
w = Window.partitionBy('mpid').orderBy(col('score').desc())
recs_ranked = recs_mapped.withColumn('rank', row_number().over(w))

als_recommendations = (
    recs_ranked
    .groupBy('mpid')
    .agg(
        collect_list(struct(col('rank'), col('item'), col('score')))
            .alias('recs_raw')
    )
    .selectExpr(
        'mpid',
        "transform(array_sort(recs_raw, (l, r) -> l.rank - r.rank), x -> x.item) as recommendations",
        "transform(array_sort(recs_raw, (l, r) -> l.rank - r.rank), x -> ROUND(x.score, 6)) as scores"
    )
)

als_recommendations.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(recs_table)

print(f'Saved {als_recommendations.count():,} user recommendation sets to {recs_table}')
display(spark.table(recs_table).limit(5))

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Derive constraint name from the table config (schema-scoped, must be unique)
recs_table_short = recs_table.split('.')[-1]
pk_constraint = f'{recs_table_short}_pk'

# Enable CDF for online store sync
spark.sql(f"""ALTER TABLE {recs_table}
             SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')""")

# Add primary key constraint on mpid
spark.sql(f"""ALTER TABLE {recs_table} ALTER COLUMN mpid SET NOT NULL""")
spark.sql(f"""ALTER TABLE {recs_table} DROP CONSTRAINT IF EXISTS {pk_constraint}""")
spark.sql(f"""ALTER TABLE {recs_table} ADD CONSTRAINT {pk_constraint} PRIMARY KEY (mpid)""")

# Create or reuse an online store
try:
    store = fe.get_online_store(name=online_store_name)
    print(f'Online store already exists: {online_store_name} (state: {store.state})')
except Exception:
    try:
        print(f'Creating online store: {online_store_name} ...')
        fe.create_online_store(name=online_store_name, capacity='CU_1')
        store = fe.get_online_store(name=online_store_name)
        print(f'Online store created (state: {store.state})')
    except Exception as e:
        store = fe.get_online_store(name=online_store_fallback)
        print(f'Quota reached, reusing shared store: {online_store_fallback} (state: {store.state})')

# Publish to online store
fe.publish_table(
    name=recs_table,
    online_store=store,
    online_table_name=online_table_name,
)
print(f'Published {recs_table} → {online_table_name}')

In [0]:
# Quick verification: read back from the Delta table and show a sample user
sample = spark.table(recs_table).limit(1).collect()[0]

print(f'Sample lookup — mpid: {int(sample.mpid)}')
print(f'  Top-5 recommendations:')
for i, (product, score) in enumerate(zip(sample.recommendations[:5], sample.scores[:5]), 1):
    print(f'    {i}. {product}  (score: {score})')

print(f'\n  Total recommendations: {len(sample.recommendations)}')
print(f'\nOnline table: {online_table_name}')
print(f'Online store: {online_store_name}')

---
### 3c — Lakebase-backed PyFunc for Model Serving
The same pattern as the MBA notebook's `MBARecommenderModel`:

1. **`load_context`** — reads the pre-computed `als_recommendations` table from the Lakebase-backed online store at model startup
2. **`predict`** — takes one or more `mpid` values and returns ranked recommendations via an O(1) lookup
3. The frontend hits a **single Model Serving endpoint** — no need to know about Lakebase, ALS, or integer mappings

| | MBA (`01`) | ALS (`02`) |
|---|---|---|
| **Input** | Cart (list of products) | User ID (`mpid`) |
| **`load_context`** | Fetches association rules from online table | Fetches pre-computed recommendations from online table |
| **`predict`** | Cross-join cart × rules → score → top-k | Lookup by `mpid` → return pre-computed top-k |
| **Latency** | Proportional to \|rules\| × \|cart\| | O(1) per user |
| **When to use** | Guest / anonymous users (no `mpid`) | Logged-in users (known `mpid`) |

In [0]:
import json
import pandas as pd
import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.types import ColSpec, Schema, DataType
from mlflow.types.schema import Array


class ALSRecommenderModel(mlflow.pyfunc.PythonModel):
    """
    Lakebase-backed PyFunc for ALS collaborative filtering.

    Mirrors the LakebaseRecommenderModel pattern from the MBA notebook:
    - load_context: reads pre-computed recommendations from the online table
    - predict: looks up ranked recommendations by mpid
    """

    def __init__(self, k=20):
        self.k = k

    def load_context(self, context):
        """Fetch pre-computed recommendations from the Lakebase-backed table."""
        with open(context.artifacts['config']) as f:
            cfg = json.load(f)

        # ---- Production (Model Serving): use databricks-sql-connector ----
        # from databricks import sql
        # with sql.connect(
        #     server_hostname=cfg['server_hostname'],
        #     http_path=cfg['http_path']
        # ) as conn:
        #     cursor = conn.cursor()
        #     cursor.execute(
        #         f"SELECT mpid, recommendations, scores "
        #         f"FROM {cfg['recommendations_table']}"
        #     )
        #     self.recs_df = cursor.fetchall_arrow().to_pandas()

        # ---- Notebook testing: Spark is available ----
        from pyspark.sql import SparkSession
        spark = SparkSession.getActiveSession()
        self.recs_df = spark.table(cfg['recommendations_table']).toPandas()
        # Index by mpid for O(1) lookup
        self.recs_df['mpid'] = self.recs_df['mpid'].astype(int)
        self.recs_lookup = self.recs_df.set_index('mpid')

    def predict(self, context, model_input):
        """Look up pre-computed recommendations for each mpid."""
        results = []
        for _, row in model_input.iterrows():
            mpid = int(row['mpid'])
            if mpid in self.recs_lookup.index:
                user_row = self.recs_lookup.loc[mpid]
                results.append({
                    'mpid': mpid,
                    'recommendations': list(user_row['recommendations'][:self.k]),
                    'scores': list(user_row['scores'][:self.k])
                })
            else:
                # Unknown user — return empty (could fall back to popularity)
                results.append({
                    'mpid': mpid,
                    'recommendations': [],
                    'scores': []
                })
        return pd.DataFrame(results)

In [0]:
import mlflow
import json
import os

os.makedirs('artifacts', exist_ok=True)
mlflow.set_experiment(experiment_name)

# Config artifact tells load_context which table to query
config = {'recommendations_table': recs_table}
with open('artifacts/als_lakebase_config.json', 'w') as f:
    json.dump(config, f)

# Define input / output schema
input_schema = Schema([
    ColSpec(DataType.long, 'mpid')
])
output_schema = Schema([
    ColSpec(DataType.long, 'mpid'),
    ColSpec(Array(DataType.string), 'recommendations'),
    ColSpec(Array(DataType.double), 'scores')
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Sample input
model_input = pd.DataFrame({'mpid': [1]})

# Log and register
with mlflow.start_run(run_name='als_lakebase_pyfunc'):
    mlflow.pyfunc.log_model(
        'model',
        python_model=ALSRecommenderModel(),
        registered_model_name=model_name,
        artifacts={'config': 'artifacts/als_lakebase_config.json'},
        input_example=model_input,
        signature=signature
    )
print(f'Model registered as {model_name}')

In [0]:
import mlflow
import pandas as pd

# Load model from latest run and generate recommendations
details = mlflow.last_active_run()
model_uri = f"runs:/{details.info.run_id}/model"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Test with sample user IDs — grab a few from the recommendations table
sample_mpids = (
    spark.table(recs_table)
    .select('mpid')
    .limit(3)
    .toPandas()
)

input_data = pd.DataFrame({'mpid': sample_mpids['mpid'].tolist() + [999999]})
print(f'Input mpids: {input_data["mpid"].tolist()}')
print(f'  (999999 is an unknown user — should return empty recommendations)\n')

results = loaded_model.predict(input_data)
for _, row in results.iterrows():
    mpid = int(row['mpid'])
    recs = row['recommendations']
    scores = row['scores']
    print(f'mpid {mpid}:')
    if recs:
        for i, (product, score) in enumerate(zip(recs[:5], scores[:5]), 1):
            print(f'  {i}. {product}  (score: {score})')
        print(f'  ... ({len(recs)} total recommendations)')
    else:
        print('  (no recommendations — unknown user)')